# 📸 Day 03a: HLD — Instagram

---

## 🎯 What You'll Master Today
- Photo upload & storage (S3 + CDN)
- Feed generation pipeline
- Stories architecture
- Explore / recommendation

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  INSTAGRAM — ARCHITECTURE OVERVIEW                               ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Core features: Post photos, Feed, Stories, Explore, Like/Comment║
║                                                                   ║
║  Upload Path:                                                    ║
║  Client ──► Upload Service ──► S3 (original)                    ║
║                    │                  │                           ║
║                    │            Image Processing                  ║
║                    │            (resize, filter, thumbnail)       ║
║                    │                  │                           ║
║                    └──► Post metadata ──► DB                     ║
║                                      └──► CDN (serve images)    ║
║                                                                   ║
║  Read Path:                                                      ║
║  Client ──► Feed Service ──► Feed Cache (Redis)                 ║
║                                  │                                ║
║                          Image URLs → CDN → Client               ║
║                                                                   ║
║  Key difference from Twitter:                                    ║
║  • Media-heavy (images 200KB-5MB each)                          ║
║  • Storage is the bottleneck, not compute                       ║
║  • CDN is CRITICAL (serve images close to users)                ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Scale Estimation
# ============================================================

print("""
📊 INSTAGRAM SCALE ESTIMATION

Assumptions:
  • DAU: 500M users
  • 20% post daily, average 1 post/day = 100M uploads/day
  • Average photo: 500KB (after compression)
  • Read:Write = 50:1

Upload QPS:
  100M / 86400 ≈ 1,200 uploads/sec

Feed reads:
  500M × 10 feed views/day / 86400 ≈ 58,000 reads/sec

Storage (1 year — photos only):
  100M/day × 500KB × 365 ≈ 18 PB/year!
  + thumbnails (multiple sizes) → ~25 PB/year

Bandwidth:
  Upload: 1,200/sec × 500KB = 600 MB/sec ingress
  Download: 58,000/sec × 5 images × 200KB = 58 GB/sec egress
  → CDN absorbs most egress!

💡 This is why Instagram uses S3 + CloudFront CDN
""")

In [ ]:
# ============================================================
# 2. Photo Upload Pipeline
# ============================================================

print("""
📤 PHOTO UPLOAD PIPELINE

Step 1: Client uploads photo
  • Direct upload to S3 via pre-signed URL (bypass API server)
  • Pre-signed URL flow:
    Client → API → generate pre-signed URL → return to client
    Client → S3 (direct upload with pre-signed URL)
    Client → API → confirm upload, save metadata

Step 2: Image Processing (async via message queue)
  • Generate thumbnails: 150×150, 320×320, 640×640, 1080×1080
  • Apply filters (if any)
  • Strip EXIF data (privacy!)
  • Store all versions in S3

Step 3: CDN Distribution
  • S3 origin → CDN edge servers worldwide
  • CDN caches popular images
  • URL format: cdn.instagram.com/{hash}_{size}.jpg

Step 4: Fan-out to followers
  • Same as Twitter: push post_id to follower feed caches
  • Feed cache stores post_ids, not actual images

Why pre-signed URL?
  • API server doesn't handle large file transfer
  • Reduces API server load
  • S3 handles multipart upload natively
  • Pre-signed URL expires after 15 min (security)
""")

In [ ]:
# ============================================================
# 3. Data Models
# ============================================================

print("""
📦 DATA MODELS

Posts Table (SQL):
┌─────────────────────────────────────────────────────┐
│ post_id   │ user_id │ caption │ photo_url │ ts     │
│ (Snowflake)│ FK     │ TEXT    │ S3 key    │ INDEX  │
└─────────────────────────────────────────────────────┘

Photos Table (multiple sizes):
┌──────────────────────────────────────┐
│ photo_id │ post_id │ size │ s3_key   │
│          │ FK      │ ENUM │ VARCHAR  │
│          │         │ thumb│          │
│          │         │ med  │          │
│          │         │ full │          │
└──────────────────────────────────────┘

Likes (Counter + who liked):
  Option A: Row per like → easy to query who liked
  ┌──────────────────────────────────┐
  │ post_id │ user_id │ created_at   │
  │ (composite PK)    │              │
  └──────────────────────────────────┘
  
  Option B: Counter cache in Redis → fast count
  Key: likes:{post_id} → SET of user_ids
  SCARD(likes:{post_id}) → count O(1)

Comments:
  ┌──────────────────────────────────────┐
  │ comment_id│ post_id│ user_id│ text   │
  └──────────────────────────────────────┘

Stories (TTL-based):
  • Same as posts but with 24hr TTL
  • Stored in Redis with EXPIRE
  • After expiry: move to archive (S3 Glacier)
""")

In [ ]:
# ============================================================
# 4. Full Architecture
# ============================================================

print("""
╔════════════════════════════════════════════════════════════════╗
║  INSTAGRAM FULL ARCHITECTURE                                 ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  ┌─────────┐                                                  ║
║  │ Client  │───────────────────────────────────┐              ║
║  └────┬────┘                                   │ (upload)     ║
║       │ (API)                             ┌────▼────┐        ║
║  ┌────▼────┐                              │   S3    │        ║
║  │   LB    │                              │(photos) │        ║
║  └────┬────┘                              └────┬────┘        ║
║  ┌────▼──────────────────┐                     │              ║
║  │    API Servers        │                ┌────▼────┐        ║
║  │  ┌──────┬──────┐     │                │  CDN    │        ║
║  │  │Upload│Feed  │     │                │(serve)  │        ║
║  │  │Svc   │Svc   │     │                └─────────┘        ║
║  │  └───┬──┴──┬───┘     │                                    ║
║  └──────┼─────┼─────────┘                                    ║
║    ┌────▼─┐ ┌─▼──────┐                                      ║
║    │Kafka │ │ Redis  │                                       ║
║    │(async)│ │Feed   │                                       ║
║    └──┬───┘ │Cache   │                                       ║
║    ┌──▼──┐  └────────┘                                       ║
║    │Image│                                                    ║
║    │Proc │  ┌──────────┐                                     ║
║    │Svc  │  │  SQL DB  │                                     ║
║    └─────┘  │(users,   │                                     ║
║             │ posts,   │                                     ║
║             │ follows) │                                     ║
║             └──────────┘                                     ║
║                                                                ║
║  Key Services:                                                ║
║  • Upload: pre-signed URL → S3 → Kafka → Image Processing   ║
║  • Feed: hybrid fan-out → Redis sorted sets                  ║
║  • Stories: Redis with 24hr TTL auto-expiry                  ║
║  • Explore: ML recommendation service (collaborative filter) ║
║  • Search: Elasticsearch (hashtags, users, locations)        ║
║  • Notifications: Firebase/APNs push                         ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Design Instagram | Photo upload (S3+CDN), hybrid fan-out feed, stories (TTL) |
| 2 | How to handle photo upload? | Pre-signed URL → S3 direct. Async image processing via Kafka |
| 3 | Feed vs Twitter? | Same hybrid approach. But images served from CDN, feed stores post_ids only |
| 4 | Stories design? | Redis with 24hr TTL. Expire → archive to S3 Glacier |
| 5 | Storage at scale? | ~25 PB/year. S3 tiered storage (hot→warm→cold). Multiple image sizes |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Storage-heavy system (~25 PB/year photos)             │
## │  • Pre-signed URLs → direct S3 upload                    │
## │  • Async image processing (resize, thumbnail, filter)    │
## │  • CDN is critical for serving images globally           │
## │  • Stories = posts with 24hr TTL in Redis                │
## │  • Feed same as Twitter: hybrid fan-out                  │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **HLD — WhatsApp Chat System**